In [1]:
PROJECT_RANDOM_STATE = 2627
print("PROJECT_RANDOM_STATE =", PROJECT_RANDOM_STATE)

PROJECT_RANDOM_STATE = 2627


# Practice 4 — Evaluation and comparison (R4)

**Scope:** canonical folds for the grid searches, shared metric code, `comparativa.csv` and the paired per-fold differences of P4.5.

> **Team code style (delete this cell before submitting).** The team must be able to defend every line.
>
> - Prefer tools seen in class or named in the statement. If you cannot explain a line in one sentence, rewrite it.
> - Plain `for` loops and `if`s, one step per line, clear variable names. No lambdas, classes or long chains.
> - Same results, simpler code. After simplifying, restart, run all and check that the numbers did not change.
> - Interpretation goes in Markdown cells, in English. Check your notebook with `revisor_notebook.py`.
> - Every number written in a Markdown cell must appear in a printed output of the notebook.
>
> **Shared names, so that the notebooks can be joined:** `dev` (development table), `X`, `y` (cyber target), `z` (storage target), `latency_ns`, `NOMINAL`, `NUMERIC`, `FEATURES`, `make_preprocessor()`, `folds` (list of `(train_idx, val_idx)`, also passed as `cv=folds`), `CLASS_NAMES` (cyber).

## Setup

In [2]:
# TODO: imports

import numpy as np
import pandas as pd
import json
from sklearn import metrics


# Challenge A — cyber

## Setup from R1 (cyber)

Paste here R1's cyber data cell (`dev`, `X`, `y`). It is **not** pasted into the final notebook.

In [3]:
#TEMPORARY cell

#read the data
dev = pd.read_csv("../datos/cyber/cyber_development.csv")
groups = pd.read_csv("../datos/cyber/cyber_groups.csv")


#classify the features
NOMINAL = ["proto","service", "conn_state"]
NUMERIC = ["duration","src_bytes","dst_bytes","missed_bytes","src_pkts","src_ip_bytes","dst_pkts","dst_ip_bytes"]

#merge the features
FEATURES = NOMINAL + NUMERIC

#define the features and the target variable
X = dev[FEATURES]
y = dev["attack_code"].to_numpy()


#lets translate the attack codes to their type
CLASS_NAMES = {}
with open("../datos/cyber/cyber_class_codes.json") as file:
    codes = json.load(file)

    #fill the dictionary with each code assigned to an attack type
    for code in codes:
        CLASS_NAMES[int(code)] = codes[code]
    
            

print(CLASS_NAMES)       
    
    




{0: 'backdoor', 1: 'ddos', 2: 'dos', 3: 'injection', 4: 'mitm', 5: 'normal', 6: 'password', 7: 'ransomware', 8: 'scanning', 9: 'xss'}


## Canonical folds (publish for R2)

In [4]:
# TODO: read the course fold file and define folds, the list passed as cv=folds. Read it, do not rebuild it.
fold_table = pd.read_csv("../folds/cyber_folds.csv")
#the positions only make sense if the fold file lists the rows in the same order as dev
assert np.array_equal(fold_table["row_id"].to_numpy(), dev["row_id"].to_numpy())
fold_id = fold_table["fold_id"].to_numpy()

#lets build the folds
folds = []
for f in range(5):
    train_idx = np.where(fold_id != f)[0]
    val_idx = np.where(fold_id == f)[0]
    folds.append((train_idx,val_idx))

print(len(folds))



5


In [5]:
# TODO: check the grouping invariant


grps = groups["feature_group"].to_numpy()
#the same for the groups file
assert np.array_equal(groups["row_id"].to_numpy(), dev["row_id"].to_numpy())
#we check that the patterns are in different groups in train and in validation
for train_idx, val_idx in folds:
    grps_train = grps[train_idx]
    grps_train = set(grps_train)
    grps_val = grps[val_idx]
    grps_val = set(grps_val) #we created two different sets to see the distinct groups in each one
    assert(len(grps_train & grps_val) == 0) #here we check that the intersection between the two groups is 0
    print(len(train_idx),len(val_idx))

print("no feature group is in both training and validation")




27942 6986
27942 6986
27943 6985
27942 6986
27943 6985
no feature group is in both training and validation


In [6]:
# TODO: metric code: macro F1 per fold, mean and spread, weighted F1, accuracy
def spread(values):
    res = np.max(values) - np.min(values)
    
    return res

def cyber_metrics(y_true, y_pred):
    macro_f1 = metrics.f1_score(y_true,y_pred,average="macro")
    weighted_f1 = metrics.f1_score(y_true,y_pred,average="weighted")
    accuracy = metrics.accuracy_score(y_true,y_pred)


    return [macro_f1, weighted_f1, accuracy]


In [7]:
print(spread([0.9436, 0.9473, 0.9490, 0.9428, 0.9451]))

0.006199999999999983


## Paired comparison against Practice 3 and Practice 2

In [8]:
# TODO: read the P2 KNN and P3 logistic per-fold scores from comparativa.csv
# and compute delta = tree - previous for each, with mean and spread
# five per-fold primary scores of one earlier row of comparativa.csv
def previous_scores(milestone, challenge, family):
    table = pd.read_csv("../comparativa.csv")
    # keep only the row with this milestone, challenge and family
    row = table[(table["milestone"] == milestone) & (table["challenge"] == challenge) & (table["family"] == family)]
    # the five scores are stored as one text, e.g. "0.9436 0.9473 0.9490 0.9428 0.9451"
    text = row["primary_metric_per_fold"].iloc[0]
    # cut the text at the spaces and turn each piece into a number
    return np.array(text.split(), dtype=float)




p2_cyber = previous_scores("P2", "cyber", "KNN")
p3_cyber = previous_scores("P3", "cyber", "LogisticRegression")
print("P2 KNN:   ", p2_cyber)
print("P3 logreg:", p3_cyber)

# check of the method with rows we already have: P3 - P2 must give
# the differences printed in Practice 3 (mean -0.1074)
deltas = p3_cyber - p2_cyber
print("P3 - P2:  ", np.round(deltas, 4))
print("mean:", round(np.mean(deltas), 4), "| spread:", round(spread(deltas), 4))

# TODO when R2 publishes the five macro F1 of the tree:
# tree_cyber = np.array([...])
# delta = tree - previous (positive means the tree did better), against p3_cyber and p2_cyber

P2 KNN:    [0.9436 0.9473 0.949  0.9428 0.9451]
P3 logreg: [0.8407 0.8332 0.8453 0.8353 0.8361]
P3 - P2:   [-0.1029 -0.1141 -0.1037 -0.1075 -0.109 ]
mean: -0.1074 | spread: 0.0112


**Reading of the differences, and which kind of comparison this is.** TODO (waiting for R2's five macro F1 of the tree)

# Challenge B — storage

## Setup from R1 (storage)

Paste here R1's storage data cell (`dev`, `X`, `z`, `latency_ns`). It is **not** pasted into the final notebook.

In [9]:
#TEMPORARY cell: change it for R1's storage data cell when it is published

#read the data
dev = pd.read_csv("../datos/storage/storage_development.csv")
groups = pd.read_csv("../datos/storage/storage_groups.csv")

#classify the features
NOMINAL = ["load_type", "io_type", "raid", "device_type"]
NUMERIC = ["block_size", "n_jobs", "iodepth", "read_fraction", "n_disks"]

#merge the features
FEATURES = NOMINAL + NUMERIC

#define the features and the target variable (z = log10 of the latency)
X = dev[FEATURES]
latency_ns = dev["latency_ns"].to_numpy()
z = np.log10(latency_ns)

## Canonical folds and metrics (publish for R3)

In [10]:
fold_table = pd.read_csv("../folds/storage_folds.csv")
#the positions only make sense if the fold file lists the rows in the same order as dev
assert np.array_equal(fold_table["row_id"].to_numpy(), dev["row_id"].to_numpy())
fold_id = fold_table["fold_id"].to_numpy()

#lets build the folds (this list is what R3 passes as cv=folds)
folds = []
for f in range(5):
    train_idx = np.where(fold_id != f)[0]
    val_idx = np.where(fold_id == f)[0]
    folds.append((train_idx, val_idx))

print(len(folds))

5


In [11]:
grps = groups["config_group"].to_numpy()
#the same for the groups file
assert np.array_equal(groups["row_id"].to_numpy(), dev["row_id"].to_numpy())
#we check that the 5 replicates of a configuration never end up split between train and validation
for train_idx, val_idx in folds:
    grps_train = set(grps[train_idx])
    grps_val = set(grps[val_idx])
    assert len(grps_train & grps_val) == 0 #the two sets of groups must not share any group
    print(len(train_idx), len(val_idx))

print("no config group is in both training and validation")

9750

 2440
9750 2440
9750 2440
9755 2435
9755 2435
no config group is in both training and validation


In [12]:
#storage metrics on the validation rows of one fold
#the first three are computed on z; MAE_ns only after turning z_pred back into nanoseconds
def storage_metrics(z_true, z_pred, latency_true):
    mae = metrics.mean_absolute_error(z_true, z_pred)
    rmse = np.sqrt(metrics.mean_squared_error(z_true, z_pred))
    r2 = metrics.r2_score(z_true, z_pred)
    mae_ns = metrics.mean_absolute_error(latency_true, 10 ** z_pred)

    return [mae, rmse, r2, mae_ns]

## Paired comparison against Practice 3 and Practice 2

In [13]:
p2_storage = previous_scores("P2", "storage", "KNN")
p3_storage = previous_scores("P3", "storage", "LinearRegression")
print("P2 KNN:   ", p2_storage)
print("P3 linear:", p3_storage)

# check of the method: P2 - P3 must give the differences printed in Practice 3 (mean -0.0181)
# for MAE(z) lower is better, so delta = previous - new
deltas = p2_storage - p3_storage
print("P2 - P3:  ", np.round(deltas, 4))
print("mean:", round(np.mean(deltas), 4), "| spread:", round(spread(deltas), 4))

# TODO when R3 publishes the five MAE(z) of the tree:
# tree_storage = np.array([...])
# delta = previous - tree (positive means the tree did better), against p3_storage and p2_storage

P2 KNN:    [0.1816 0.187  0.1778 0.1775 0.1762]
P3 linear: [0.2047 0.1937 0.1984 0.1953 0.1987]
P2 - P3:   [-0.0231 -0.0067 -0.0206 -0.0178 -0.0225]
mean: -0.0181 | spread: 0.0164


**Reading of the differences, and which kind of comparison this is.** TODO (waiting for R3's five MAE(z) of the tree)

# Comparison table

In [14]:
# TODO (waiting for R2's and R3's results): append the Practice 4 rows to comparativa.csv, with the route column,
# leaving the Practice 2 and Practice 3 rows unchanged

## Penalty check (R4 veto)

TODO (when R2 and R3 publish their notebooks): check them against the list of automatic penalties of the statement.